# East New York rezoning — building-level (BIN) table

Same idea as the Jerome Avenue notebook (one row per **BIN**), but the input is two raw
BBL lists in `data/raw/east new york bbls/` — `verified_bbls.txt` and `unverified_bbls.txt`.

Each listed BBL is resolved one of three ways, recorded in the **`resolution`** column:
- **`listed_lot`** — an individual lot that is in current PLUTO (the verified ones, and most
  unverified ones).
- **`block_expanded`** — a *block-level* BBL (lot `0000`, i.e. a whole tax block). These are
  not real lots and aren't in PLUTO, so we enumerate every current lot on that block and pull
  the buildings on them — "the BINs in them".
- **`missing_lot_footprint`** — an individual lot not in current PLUTO, but a building
  footprint still references it (a retired/merged lot); we keep its BIN as a historical trace.

We also carry **`last_status_type`** from the footprint layer so demolished/historical
buildings are visible rather than silently dropped. Sparse, with sanity prints.

In [1]:
import pandas as pd
import requests
import geopandas as gpd

DOMAIN = "data.cityofnewyork.us"


def soda(dataset_id, params):
    """Query a Socrata dataset (anonymous) and return a DataFrame."""
    url = f"https://{DOMAIN}/resource/{dataset_id}.json"
    r = requests.get(url, params=params, timeout=300)
    r.raise_for_status()
    return pd.DataFrame(r.json())


def in_clause(values):
    return "'" + "','".join(values) + "'"


def soda_in(dataset_id, select, field, values, batch=250, extra=None):
    """Query `dataset_id` for `field in (values)`, chunked to beat URL-length limits."""
    values = list(dict.fromkeys(values))
    out = []
    for i in range(0, len(values), batch):
        where = f"{field} in ({in_clause(values[i:i + batch])})"
        if extra:
            where = f"{extra} AND {where}"
        chunk = soda(dataset_id, {"$select": select, "$where": where, "$limit": "200000"})
        if len(chunk):
            out.append(chunk)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

## 1. Load the two BBL lists
BBLs are 10-digit strings (borough 3 = Brooklyn). A BBL whose lot is `0000` is a whole-block
reference, not an individual lot.

In [2]:
DIR = "../data/raw/east new york bbls"


def load_bbls(name):
    with open(f"{DIR}/{name}") as f:
        return [ln.strip() for ln in f if ln.strip()]


ver, unv = load_bbls("verified_bbls.txt"), load_bbls("unverified_bbls.txt")
inp = pd.DataFrame({"bbl": ver + unv,
                    "input_list": ["verified"] * len(ver) + ["unverified"] * len(unv)})
inp = (inp.groupby("bbl")["input_list"]
          .agg(lambda s: "both" if s.nunique() > 1 else s.iloc[0]).reset_index())
inp["is_block"] = inp["bbl"].str.endswith("0000")
INP_MAP = dict(zip(inp["bbl"], inp["input_list"]))


def block_ref(bbl):
    """The block-level BBL that a lot belongs to: boro + block + '0000'."""
    return bbl[0] + bbl[1:6] + "0000"


def lot_input(bbl):
    """Which input list a resolved lot traces back to (its own entry, or its block's)."""
    return INP_MAP.get(bbl) or INP_MAP.get(block_ref(bbl))


print("verified:", len(ver), "| unverified:", len(unv), "| unique BBLs:", len(inp))
print("block-level (lot 0000):", int(inp["is_block"].sum()),
      "| individual lots:", int((~inp["is_block"]).sum()))

verified: 50 | unverified: 429 | unique BBLs: 479
block-level (lot 0000): 128 | individual lots: 351


## 2. Resolve the individual lots against current PLUTO
The verified lots should all match; unverified individual lots mostly do.

In [3]:
PLUTO_COLS = "bbl,address,bldgclass,landuse,yearbuilt,unitsres,unitstotal,numbldgs,assesstot"

lot_bbls = inp.loc[~inp["is_block"], "bbl"].tolist()
pluto_lots = soda_in("64uk-42ks", PLUTO_COLS, "bbl", lot_bbls, batch=400)
if len(pluto_lots):
    pluto_lots["bbl"] = pluto_lots["bbl"].astype(str).str.split(".").str[0]
pluto_lots["resolution"] = "listed_lot"

matched = set(pluto_lots["bbl"])
missing_lots = sorted(set(lot_bbls) - matched)
for lst in ["verified", "unverified"]:
    b = set(inp.loc[(~inp["is_block"]) & (inp["input_list"].isin([lst, "both"])), "bbl"])
    print(f"{lst:10s} individual lots: {len(b):4d} | in PLUTO {len(b & matched):4d} "
          f"| missing {len(b - matched):3d}")
print("total individual-lot misses:", len(missing_lots))

verified   individual lots:   14 | in PLUTO   14 | missing   0
unverified individual lots:  337 | in PLUTO  322 | missing  15
total individual-lot misses: 15


## 3. Resolve the block-level BBLs — enumerate their lots
For each `0000` block we pull every current Brooklyn lot on that tax block.

In [4]:
block_bbls = inp.loc[inp["is_block"], "bbl"].tolist()
blocks = sorted({int(b[1:6]) for b in block_bbls})

pluto_block = []
for i in range(0, len(blocks), 200):
    sub = ",".join(str(x) for x in blocks[i:i + 200])
    chunk = soda("64uk-42ks", {"$select": PLUTO_COLS,
                               "$where": f"borough='BK' AND block in ({sub})",
                               "$limit": "500000"})
    if len(chunk):
        pluto_block.append(chunk)
pluto_block = pd.concat(pluto_block, ignore_index=True) if pluto_block else pd.DataFrame()
if len(pluto_block):
    pluto_block["bbl"] = pluto_block["bbl"].astype(str).str.split(".").str[0]
pluto_block["resolution"] = "block_expanded"
print("block-level BBLs:", len(block_bbls), "-> unique blocks:", len(blocks),
      "-> lots enumerated:", len(pluto_block))

block-level BBLs: 128 -> unique blocks: 128 -> lots enumerated: 3340


## 4. Combine resolved lots (a listed lot wins over the same lot via block-expansion)

In [5]:
pluto = pd.concat([pluto_lots, pluto_block], ignore_index=True).drop_duplicates("bbl", keep="first")
pluto["input_list"] = pluto["bbl"].map(lot_input)
for c in ["yearbuilt", "unitsres", "unitstotal", "numbldgs", "assesstot"]:
    pluto[c] = pd.to_numeric(pluto[c], errors="coerce")
print("resolved lots:", len(pluto), "| by resolution:", pluto["resolution"].value_counts().to_dict())

resolved lots: 3676 | by resolution: {'block_expanded': 3340, 'listed_lot': 336}


## 5. Building footprints (BINs) for every resolved lot — plus the missing individual lots
We include `last_status_type`, and try footprints even for lots PLUTO dropped (recovers
historical BINs on retired/merged lots). Filtering on `base_bbl` never scans the full set.

In [6]:
fp_bbls = sorted(set(pluto["bbl"]) | set(missing_lots))
fp = soda_in("5zhs-2jue", "base_bbl,bin,the_geom,construction_year,last_status_type",
             "base_bbl", fp_bbls, batch=300)
fp["base_bbl"] = fp["base_bbl"].astype(str).str.split(".").str[0]
fp["bin"] = fp["bin"].astype(str)
fp["construction_year"] = pd.to_numeric(fp["construction_year"], errors="coerce")

_geom = gpd.GeoDataFrame.from_features(
    [{"type": "Feature", "geometry": g, "properties": {}} for g in fp["the_geom"]],
    crs="EPSG:4326",
).geometry
fp["the_geom_wkt"] = _geom.to_wkt().values
fp = fp.drop(columns="the_geom")
print("BIN rows:", len(fp), "| lots with a BIN:", fp["base_bbl"].nunique())
print("footprint status:", fp["last_status_type"].value_counts(dropna=False).to_dict())

BIN rows: 3929 | lots with a BIN: 3304
footprint status: {'Constructed': 3895, 'Alteration': 22, 'Split': 5, nan: 4, 'Correction': 1, 'Merged': 1, 'Initialization': 1}


## 6. Attach PLUTO to each BIN
BINs on a lot PLUTO doesn't have are tagged `missing_lot_footprint` (historical trace).

In [7]:
bins = fp.merge(pluto, left_on="base_bbl", right_on="bbl", how="left").drop(columns="bbl")
bins["resolution"] = bins["resolution"].fillna("missing_lot_footprint")
bins["input_list"] = bins["input_list"].where(bins["input_list"].notna(),
                                               bins["base_bbl"].map(lot_input))
print("BIN rows:", len(bins), "| BINs with no PLUTO match:", int(bins["bldgclass"].isna().sum()))

BIN rows: 3929 | BINs with no PLUTO match: 1


## 7. Apartment flag + units (PLUTO-primary)
Building class **C = walk-up / D = elevator** apartments; `res_units` = PLUTO `unitsres`.

In [8]:
bins["res_units"] = bins["unitsres"]
bins["is_apartment"] = (bins["bldgclass"].fillna("").str[0].isin(["C", "D"])
                        | (bins["res_units"] >= 3))
print("apartment BINs:", int(bins["is_apartment"].sum()), "of", len(bins),
      "| residential units:", int(bins["res_units"].fillna(0).sum()))

apartment BINs: 1130 of 3929 | residential units: 22970


## 8. DOB job filings — apartment cross-check and New-Building dates
Two systems: legacy **BIS** (`ic3t-wcy2`, m/d/Y dates, occupancy J-2 / R-2) and **DOB NOW**
(`w9ak-ipjd`, ISO dates) for post-~2021 buildings. `dob_nb_filed` = when new apartments were
*expected* (first New-Building filing); `dob_nb_signoff` = DOB sign-off.

In [9]:
APT_OCC = {"J-2", "R-2"}
bin_list = fp["bin"].unique().tolist()

dob = soda_in("ic3t-wcy2",
              ("bin__,job_type,proposed_occupancy,existing_occupancy,"
               "proposed_dwelling_units,existing_dwelling_units,pre__filing_date,signoff_date"),
              "bin__", bin_list, batch=250)
for c in ["proposed_dwelling_units", "existing_dwelling_units"]:
    dob[c] = pd.to_numeric(dob[c], errors="coerce")
for c in ["pre__filing_date", "signoff_date"]:
    dob[c] = pd.to_datetime(dob[c], format="%m/%d/%Y", errors="coerce")
apt_occ = dob["proposed_occupancy"].isin(APT_OCC) | dob["existing_occupancy"].isin(APT_OCC)
apt_units = dob[["proposed_dwelling_units", "existing_dwelling_units"]].max(axis=1) >= 3
dob["is_apt_filing"] = apt_occ | apt_units
nb_bis = (dob[dob["job_type"] == "NB"]
          .rename(columns={"bin__": "bin", "pre__filing_date": "filed", "signoff_date": "signoff"})
          [["bin", "filed", "signoff"]])

now = soda_in("w9ak-ipjd", "bin,filing_date,signoff_date", "bin", bin_list, batch=250,
              extra="job_type='New Building'")
if len(now):
    now["bin"] = now["bin"].astype(str)
    now = now.rename(columns={"filing_date": "filed", "signoff_date": "signoff"})
    for c in ["filed", "signoff"]:
        now[c] = pd.to_datetime(now[c], errors="coerce")
    now = now[["bin", "filed", "signoff"]]
else:
    now = pd.DataFrame(columns=["bin", "filed", "signoff"])

nb = (pd.concat([nb_bis, now], ignore_index=True)
      .groupby("bin").agg(dob_nb_filed=("filed", "min"), dob_nb_signoff=("signoff", "max"))
      .reset_index())
dob_bin = dob.groupby("bin__").agg(
    dob_is_apartment=("is_apt_filing", "any"),
    dob_dwelling_units=("proposed_dwelling_units", "max"),
    n_dob_filings=("is_apt_filing", "size"),
).reset_index().rename(columns={"bin__": "bin"})

bins = bins.merge(dob_bin, on="bin", how="left").merge(nb, on="bin", how="left")
bins["has_dob_record"] = bins["n_dob_filings"].notna()
bins["dob_is_apartment"] = bins["dob_is_apartment"].fillna(False)
print("BINs with a DOB filing:", int(bins["has_dob_record"].sum()), "of", len(bins),
      "| with a New-Building date:", int(bins["dob_nb_filed"].notna().sum()))

BINs with a DOB filing: 1599 of 3929 | with a New-Building date: 553


## 9. Final BIN table + save + summary

In [10]:
cols = ["bin", "base_bbl", "input_list", "resolution", "last_status_type", "address",
        "construction_year", "yearbuilt", "bldgclass", "landuse", "is_apartment", "res_units",
        "unitstotal", "numbldgs", "assesstot", "dob_is_apartment", "dob_dwelling_units",
        "dob_nb_filed", "dob_nb_signoff", "n_dob_filings", "has_dob_record", "the_geom_wkt"]
bins = bins[cols]
bins.to_csv("east_ny_bins.csv", index=False)
print("saved:", bins.shape, "-> notebooks/east_ny_bins.csv")
print()
print("BINs by input list x resolution:")
print(pd.crosstab(bins["input_list"], bins["resolution"], margins=True))
print()
print("apartment BINs:", int(bins["is_apartment"].sum()),
      "| residential units:", int(bins["res_units"].fillna(0).sum()))
bins.drop(columns="the_geom_wkt").head()

saved: (3929, 22) -> notebooks/east_ny_bins.csv

BINs by input list x resolution:
resolution  block_expanded  listed_lot  missing_lot_footprint   All
input_list                                                         
unverified            2407         327                      1  2735
verified              1180          14                      0  1194
All                   3587         341                      1  3929

apartment BINs: 1130 | residential units: 22970


,bin,base_bbl,input_list,resolution,last_status_type,address,construction_year,yearbuilt,bldgclass,landuse,...,res_units,unitstotal,numbldgs,assesstot,dob_is_apartment,dob_dwelling_units,dob_nb_filed,dob_nb_signoff,n_dob_filings,has_dob_record
0,3038666,3014370008,unverified,block_expanded,Constructed,2402 ATLANTIC AVENUE,1920.0,1920.0,N9,8,...,0.0,1.0,1.0,3224700.0,True,24.0,NaT,NaT,16.0,True
1,3038667,3014370015,unverified,block_expanded,Constructed,2416 ATLANTIC AVENUE,1920.0,1920.0,H3,5,...,0.0,1.0,1.0,3721500.0,True,119.0,NaT,NaT,15.0,True
2,3330718,3014370021,unverified,block_expanded,Constructed,2432 ATLANTIC AVENUE,1926.0,1926.0,F9,6,...,0.0,1.0,2.0,307800.0,False,NaN,NaT,NaT,NaN,False
3,3330839,3014370021,unverified,block_expanded,Constructed,2432 ATLANTIC AVENUE,1926.0,1926.0,F9,6,...,0.0,1.0,2.0,307800.0,False,NaN,NaT,NaT,NaN,False
4,3413636,3014370027,unverified,block_expanded,Constructed,2460 ATLANTIC AVENUE,2012.0,2012.0,F5,6,...,0.0,1.0,1.0,220050.0,False,NaN,NaT,NaT,NaN,False
